In [1]:
import tensorflow as tf

from tensorflow.keras import layers
from tensorflow.keras import models

from tensorflow.keras.optimizers import Adam

from tensorflow.keras.callbacks import (
    EarlyStopping,
    ModelCheckpoint,
    ReduceLROnPlateau
)

I0000 00:00:1781999296.979798  132809 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1781999297.108790  132809 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1781999300.943537  132809 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


In [2]:
IMG_HEIGHT = 224
IMG_WIDTH = 224

BATCH_SIZE = 32
SEED = 42

In [7]:
dataset_path = "/workspaces/plant_disease_detection/data/raw"

In [8]:
train_dataset = tf.keras.utils.image_dataset_from_directory(

    dataset_path,

    validation_split=0.2,

    subset="training",

    seed=SEED,

    image_size=(IMG_HEIGHT, IMG_WIDTH),

    batch_size=BATCH_SIZE

)

Found 20638 files belonging to 15 classes.
Using 16511 files for training.


In [9]:
validation_dataset = tf.keras.utils.image_dataset_from_directory(

    dataset_path,

    validation_split=0.2,

    subset="validation",

    seed=SEED,

    image_size=(IMG_HEIGHT, IMG_WIDTH),

    batch_size=BATCH_SIZE

)

Found 20638 files belonging to 15 classes.
Using 4127 files for validation.


In [10]:
class_names = train_dataset.class_names

NUM_CLASSES = len(class_names)

print("Number of classes:", NUM_CLASSES)

Number of classes: 15


In [11]:
AUTOTUNE = tf.data.AUTOTUNE

train_dataset = train_dataset.prefetch(
    buffer_size=AUTOTUNE
)

validation_dataset = validation_dataset.prefetch(
    buffer_size=AUTOTUNE
)

In [12]:
data_augmentation = tf.keras.Sequential([

    layers.RandomFlip("horizontal"),

    layers.RandomRotation(0.2),

    layers.RandomZoom(0.2),

    layers.RandomContrast(0.2)

])

In [13]:
def build_cnn_model():

    model = models.Sequential([

        layers.Input(shape=(IMG_HEIGHT,
                            IMG_WIDTH,
                            3)),

        data_augmentation,

        layers.Rescaling(1./255),

        layers.Conv2D(32, (3,3), activation='relu'),
        layers.MaxPooling2D((2,2)),

        layers.Conv2D(64, (3,3), activation='relu'),
        layers.MaxPooling2D((2,2)),

        layers.Conv2D(128, (3,3), activation='relu'),
        layers.MaxPooling2D((2,2)),

        layers.Flatten(),

        layers.Dense(128, activation='relu'),

        layers.Dropout(0.3),

        layers.Dense(NUM_CLASSES,
                     activation='softmax')

    ])

    return model

In [14]:
model = build_cnn_model()

W0000 00:00:1781999632.011816  132809 cpu_allocator_impl.cc:82] Allocation of 44302336 exceeds 10% of free system memory.
W0000 00:00:1781999632.071491  132809 cpu_allocator_impl.cc:82] Allocation of 44302336 exceeds 10% of free system memory.
W0000 00:00:1781999632.086946  132809 cpu_allocator_impl.cc:82] Allocation of 44302336 exceeds 10% of free system memory.


In [15]:
model.compile(

    optimizer=Adam(
        learning_rate=0.001
    ),

    loss='sparse_categorical_crossentropy',

    metrics=['accuracy']

)

In [16]:
early_stopping = EarlyStopping(

    monitor='val_loss',

    patience=5,

    restore_best_weights=True
)

checkpoint = ModelCheckpoint(

    filepath='../artifacts/best_model.keras',

    monitor='val_accuracy',

    save_best_only=True,

    verbose=1
)

reduce_lr = ReduceLROnPlateau(

    monitor='val_loss',

    factor=0.2,

    patience=3,

    verbose=1
)

In [ ]:
EPOCHS = 30

history = model.fit(

    train_dataset,

    validation_data=validation_dataset,

    epochs=EPOCHS,

    callbacks=[

        early_stopping,

        checkpoint,

        reduce_lr
    ]
)

Epoch 1/30


W0000 00:00:1781999677.238733  132809 cpu_allocator_impl.cc:82] Allocation of 44302336 exceeds 10% of free system memory.
W0000 00:00:1781999678.019513  137818 cpu_allocator_impl.cc:82] Allocation of 19267584 exceeds 10% of free system memory.


516/516 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/step - accuracy: 0.3788 - loss: 1.9339
Epoch 1: val_accuracy improved from None to 0.66901, saving model to ../artifacts/best_model.keras

Epoch 1: finished saving model to ../artifacts/best_model.keras
516/516 ━━━━━━━━━━━━━━━━━━━━ 1453s 3s/step - accuracy: 0.5055 - loss: 1.5280 - val_accuracy: 0.6690 - val_loss: 1.0255 - learning_rate: 0.0010
Epoch 2/30
212/516 ━━━━━━━━━━━━━━━━━━━━ 15:13 3s/step - accuracy: 0.6515 - loss: 1.0639